In [2]:
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import NoSuchElementException, ElementNotInteractableException 
import time
import re

In [3]:
# 2. CSV 파일 읽기
# lifestyle-entertainment, IoT, Support, 
# AppFamilies, BizIntel, Website-AppBuilding, HumanResources, ArtificialIntelligence, Content-Files, Productivity, ITOperations, Communication, Commerce, Marketing, Sales-CRM
data = pd.read_excel("D:/code/kka_1/data/Zapier_Sales-CRM_apps.xlsx")
print(data.shape)
data.head()

(1575, 3)


,AppName,AppInfo,Url
0,Schedule by Zapier,Schedule is a native Zapier app you can use fo...,https://zapier.com/apps/schedule/integrations
1,Google Forms,Google Forms is an easy way to collect data fr...,https://zapier.com/apps/google-forms/integrations
2,Calendly,Calendly is an elegant and simple scheduling t...,https://zapier.com/apps/calendly/integrations
3,Typeform,Typeform helps you ask awesomely online! If yo...,https://zapier.com/apps/typeform/integrations
4,Salesforce,Salesforce is a leading enterprise customer re...,https://zapier.com/apps/salesforce/integrations


In [4]:
# 3. url 리스트 추출
url_list = data["Url"].tolist()
print(len(url_list), url_list[0])
app_list = data["AppName"].tolist()
print(len(app_list), app_list[0])

1575 https://zapier.com/apps/schedule/integrations
1575 Schedule by Zapier


In [5]:
driver = webdriver.Chrome()
driver.maximize_window()
for idx, url in enumerate(url_list): # 각 url 순회하며 크롤링
    print(f"\n[{idx+1}/{len(url_list)}] 현재 URL: {url}")
    driver.get(url)
    wait = WebDriverWait(driver, np.random.uniform(1,2)) # WebDriverWait 객체 생성
    button_xpath = '//*[@id="zap-template-list"]/div[2]/div/button/span/span' # button_XPATH 지정
    # 버튼이 더 이상 존재하지 않을 때까지 클릭
    while True:
        try:
            # 버튼이 존재하고 클릭 가능할 때까지 대기
            button = wait.until(EC.element_to_be_clickable((By.XPATH, button_xpath)))
            # 버튼이 화면 내에 위치하도록 스크롤 조정
            driver.execute_script("arguments[0].scrollIntoView();", button)
            time.sleep(0.5)  # 스크롤 반영 대기
            # JavaScript를 사용하여 클릭
            driver.execute_script("arguments[0].click();", button)
            # 클릭 후 페이지가 반응할 시간을 부여
            time.sleep(1)            
        except Exception as e:
            print("더 이상 클릭할 버튼 없음:", e)
            break  # 버튼이 없으면 루프 종료
    # zap 리스트 찾기
    zap_list = driver.find_elements(By.XPATH, "//ul[@class='css-1mmsjt2']/li")
    zap_data = [] # 데이터 저장 리스트
    for zap in zap_list: # 개별 Zap 정보 추출
        try:
            # Zap name 가져오기
            zap_name = zap.find_element(By.XPATH, ".//h3[contains(@class, 'css-w4g7zr-Heading-ZapCard__title')]").text
            # URL 가져오기
            url_element = zap.find_element(By.XPATH, ".//a[contains(@class, '_link_1gyux_1')]")
            zap_url = url_element.get_attribute("href")
            # Using Apps 가져오기
            using_apps = zap.find_element(By.XPATH, ".//div[contains(@class, 'css-1nzgdax-ZapCard__metaInfoArea')]").text
            # 데이터 저장
            zap_data.append({"Zap Name": zap_name, "URL": zap_url, "Using Apps": using_apps})
        except Exception as e:
            print(f"오류 발생: {e}")  # 오류 발생 시 계속 진행
    # 데이터프레임 변환 및 저장
    df = pd.DataFrame(zap_data)
    # 파일명에 사용할 수 없는 특수문자들을 ~로 대체
    safe_filename = re.sub(r'[\\/:*?"<>|]', '~', app_list[idx])
    print(app_list[idx], '->',safe_filename)
    df.to_csv(r"D:/code/kka_1/data/Zap_Template_data/SalesCRM/{}_ZapTemplateList.csv".format(safe_filename), encoding='UTF-8', index=False)
    print('{}_ZapTemplateList 파일 저장 완료.'.format(app_list[idx]))



[1/1575] 현재 URL: https://zapier.com/apps/schedule/integrations
더 이상 클릭할 버튼 없음: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF627145335+78597]
	GetHandleVerifier [0x00007FF627145390+78688]
	(No symbol) [0x00007FF626EF91AA]
	(No symbol) [0x00007FF626F4F149]
	(No symbol) [0x00007FF626F4F3FC]
	(No symbol) [0x00007FF626FA2467]
	(No symbol) [0x00007FF626F7712F]
	(No symbol) [0x00007FF626F9F2BB]
	(No symbol) [0x00007FF626F76EC3]
	(No symbol) [0x00007FF626F403F8]
	(No symbol) [0x00007FF626F41163]
	GetHandleVerifier [0x00007FF6273EEEED+2870973]
	GetHandleVerifier [0x00007FF6273E9698+2848360]
	GetHandleVerifier [0x00007FF627406973+2967875]
	GetHandleVerifier [0x00007FF62716017A+188746]
	GetHandleVerifier [0x00007FF62716845F+222255]
	GetHandleVerifier [0x00007FF62714D2B4+111236]
	GetHandleVerifier [0x00007FF62714D462+111666]
	GetHandleVerifier [0x00007FF627133589+5465]
	BaseThreadInitThunk [0x00007FFCF7967374+20]
	RtlUserThreadStart [0x00007FFCF97FCC91+33]

Schedule by Zapier -> Schedule by

# ---- 아래는 안 될때 issue 해결용 ---

In [ ]:
driver = webdriver.Chrome()
driver.get('https://zapier.com/apps/zapier-tables/integrations')
driver.maximize_window()
# WebDriverWait 객체 생성
wait = WebDriverWait(driver, np.random.uniform(1,2))

In [ ]:
# XPATH 지정
button_xpath = '//*[@id="zap-template-list"]/div[2]/div/button/span/span'

# 버튼이 더 이상 존재하지 않을 때까지 클릭
while True:
    try:
        # 버튼이 존재하고 클릭 가능할 때까지 대기
        button = wait.until(EC.element_to_be_clickable((By.XPATH, button_xpath)))

        # 버튼이 화면 내에 위치하도록 스크롤 조정
        driver.execute_script("arguments[0].scrollIntoView();", button)
        time.sleep(0.5)  # 스크롤 반영 대기

        # JavaScript를 사용하여 클릭
        driver.execute_script("arguments[0].click();", button)

        # 클릭 후 페이지가 반응할 시간을 부여
        time.sleep(1)
        
    except Exception as e:
        print("더 이상 클릭할 버튼 없음:", e)
        break  # 버튼이 없으면 루프 종료

In [ ]:
# 모든 <li class="css-1qwpe4d"> 요소 찾기
zap_list = driver.find_elements(By.XPATH, "//ul[@class='css-1mmsjt2']/li")
print(zap_list)

In [ ]:
# 데이터 저장 리스트
zap_data = []

# 개별 Zap 정보 추출
for zap in zap_list:
    try:
        # Zap name 가져오기
        zap_name = zap.find_element(By.XPATH, ".//h3[contains(@class, 'css-w4g7zr-Heading-ZapCard__title')]").text

        # URL 가져오기
        url_element = zap.find_element(By.XPATH, ".//a[contains(@class, '_link_1gyux_1')]")
        zap_url = url_element.get_attribute("href")

        # Using Apps 가져오기
        using_apps = zap.find_element(By.XPATH, ".//div[contains(@class, 'css-1nzgdax-ZapCard__metaInfoArea')]").text

        # 데이터 저장
        zap_data.append({
            "Zap Name": zap_name,
            "URL": zap_url,
            "Using Apps": using_apps
        })

    except Exception as e:
        print(f"오류 발생: {e}")  # 오류 발생 시 계속 진행

# 데이터프레임 변환 및 출력
df = pd.DataFrame(zap_data)
df


In [ ]:
# 모든 <li class="css-1qwpe4d"> 요소 찾기
zap_list = driver.find_elements(By.XPATH, "//ul[@class='css-4dc3p1-ZapTemplateList__list']/li")

# 데이터 저장 리스트
zap_data = []

# 개별 Zap 정보 추출
for zap in zap_list:
    try:
        # Zap name 가져오기
        zap_name = zap.find_element(By.XPATH, ".//h3[contains(@class, 'Heading-ZapCard__title')]").text

        # URL 가져오기
        url_element = zap.find_element(By.XPATH, ".//a[contains(@class, 'Link')]")
        zap_url = url_element.get_attribute("href")

        # Using Apps 가져오기
        using_apps = zap.find_element(By.XPATH, ".//div[contains(@class, 'ZapCard__metaInfoArea')]").text

        # 데이터 저장
        zap_data.append({
            "Zap Name": zap_name,
            "URL": zap_url,
            "Using Apps": using_apps
        })

    except Exception as e:
        print(f"오류 발생: {e}")  # 오류 발생 시 계속 진행

# 데이터프레임 변환 및 출력
df = pd.DataFrame(zap_data)
df


In [ ]:
df.to_csv("./Zap_data/RSSbyZapier_ZapTemplateList.csv", encoding='UTF-8', index=False)
